# Travel Style Destination Classifier

Colab-ready notebook scaffold. Run top to bottom after uploading `Tourist_Destinations.csv`.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 42
np.random.seed(SEED)

DATA_DIR = Path("data")
MODEL_DIR = Path("models")
DATA_DIR.mkdir(exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)


: 

## 1. Load the raw CSV

In [ ]:
raw_path = "/content/Tourist_Destinations.csv"  # In Colab, upload this file first.
df = pd.read_csv(raw_path)

print(df.shape)
df.head()


## 2. Create documented labels

The target label is created using a scoring rubric. Do not train on the score columns.

In [ ]:
cost_q25 = df["Avg Cost (USD/day)"].quantile(0.25)
cost_q50 = df["Avg Cost (USD/day)"].quantile(0.50)
cost_q75 = df["Avg Cost (USD/day)"].quantile(0.75)
visitor_median = df["Annual Visitors (M)"].median()
rating_median = df["Avg Rating"].median()

print({
    "cost_q25": cost_q25,
    "cost_q50": cost_q50,
    "cost_q75": cost_q75,
    "visitor_median": visitor_median,
    "rating_median": rating_median,
})


In [ ]:
def assign_travel_style(row):
    cost = row["Avg Cost (USD/day)"]
    rating = row["Avg Rating"]
    visitors = row["Annual Visitors (M)"]
    dest_type = row["Type"]
    season = row["Best Season"]
    unesco = row["UNESCO Site"] == "Yes"

    scores = {
        "Adventure": 0.0,
        "Relaxation": 0.0,
        "Culture": 0.0,
        "Budget": 0.0,
        "Luxury": 0.0,
        "Family": 0.0,
    }

    if dest_type == "Adventure":
        scores["Adventure"] += 4
    if dest_type == "Nature":
        scores["Adventure"] += 2
    if visitors < visitor_median:
        scores["Adventure"] += 1
    if season in ["Spring", "Autumn"]:
        scores["Adventure"] += 0.5

    if dest_type == "Beach":
        scores["Relaxation"] += 4
    if dest_type == "Nature":
        scores["Relaxation"] += 1.5
    if season in ["Summer", "Spring"]:
        scores["Relaxation"] += 1
    if visitors >= visitor_median:
        scores["Relaxation"] += 0.5

    if dest_type in ["Historical", "Religious"]:
        scores["Culture"] += 3.5
    if dest_type == "City":
        scores["Culture"] += 1
    if unesco:
        scores["Culture"] += 2
    if rating >= rating_median:
        scores["Culture"] += 0.5

    if cost <= cost_q25:
        scores["Budget"] += 4
    elif cost <= cost_q50:
        scores["Budget"] += 1.5
    if visitors < visitor_median:
        scores["Budget"] += 0.5
    if rating >= 4.0:
        scores["Budget"] += 0.5

    if cost >= cost_q75:
        scores["Luxury"] += 4
    elif cost >= cost_q50:
        scores["Luxury"] += 1.5
    if rating >= 4.7:
        scores["Luxury"] += 1.5
    if visitors >= visitor_median:
        scores["Luxury"] += 0.5
    if unesco:
        scores["Luxury"] += 0.5

    if dest_type in ["Beach", "City", "Nature"]:
        scores["Family"] += 1.5
    if rating >= 4.5:
        scores["Family"] += 1.5
    if visitors >= visitor_median:
        scores["Family"] += 1.5
    if cost_q25 < cost < cost_q75:
        scores["Family"] += 1.5
    if season in ["Summer", "Spring"]:
        scores["Family"] += 0.5
    if cost >= cost_q75:
        scores["Family"] -= 1
    if dest_type in ["Religious", "Adventure"]:
        scores["Family"] -= 0.5

    priority = ["Budget", "Luxury", "Culture", "Adventure", "Relaxation", "Family"]
    max_score = max(scores.values())
    winners = [label for label, score in scores.items() if score == max_score]
    return sorted(winners, key=priority.index)[0]


df["Travel Style"] = df.apply(assign_travel_style, axis=1)
df["Travel Style"].value_counts()


## 3. Create a mirror dataset with new labeled targets

we kept same size imbalanced data to

In [ ]:


dataset_2000 = (
    df.sample(frac=1, random_state=SEED) # Sample the entire df (frac=1) to shuffle its rows
      .reset_index(drop=True) # Reset index to have a clean, new index
)

dataset_2000.to_csv(DATA_DIR / "travel_data_labeled.csv", index=False)
dataset_2000["Travel Style"].value_counts()